# 📊 Aşama 1: Veri Keşfi ve Ön İşleme

**Türkçe E-Ticaret Yorumları — Duygu Analizi & Metin Özetleme**

Bu notebook'ta:
1. Hepsiburada veri setini yükleyeceğiz (3 farklı yöntem var)
2. Veri keşfi (EDA) — sınıf dağılımı, kelime istatistikleri, grafikler
3. Türkçe NLP ön işleme
4. Temizlenmiş veri `data/hepsiburada_clean.csv` olarak kaydedilecek

## 🔧 0) Colab Kurulumu (sadece Colab'da çalıştırın)

In [ ]:
import sys, os
IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    !git clone https://github.com/erncanyildirim/metin-ozetleme-ve-duygu-analizi.git
    %cd metin-ozetleme-ve-duygu-analizi
    !pip install -q -r requirements.txt
    print('✅ Colab kurulumu tamamlandı!')
else:
    print('Lokalde çalışıyorsunuz — pip install -r requirements.txt yapın')

## 📂 1) Veri Setini Edinme

**3 farklı yöntem var** — kendine en uygun olanı seç:

### 🅰️ Yöntem A: Manuel Yükleme (En Kolay — ÖNERİLEN)
Zip dosyasını zaten masaüstüne indirdiysen:
1. Aşağıdaki **Hücre A**'yı çalıştır → dosya yükleme butonu açılacak
2. `Hepsiburada Product Reviews and Ratings Dataset  .zip` dosyasını seç (~25MB, 1-2 dk sürer)

### 🅱️ Yöntem B: Google Drive (Hızlı, Tekrar Kullanılabilir)
Zip'i önce Google Drive'a yükle, sonra Colab'a mount et. **Hücre B**'yi çalıştır.

### 🅲 Yöntem C: Kaggle API (Otomatik)
API token gerekir. **Hücre C**'yi çalıştır.

### Hücre A — Manuel Yükleme

In [ ]:
# ↓ SADECE Yöntem A kullanıyorsan bu hücreyi çalıştır
if IS_COLAB:
    from google.colab import files
    import zipfile, shutil

    print('📤 Hepsiburada zip dosyasını yükleyin (~25MB):')
    uploaded = files.upload()

    os.makedirs('data', exist_ok=True)
    for filename in uploaded.keys():
        if filename.endswith('.zip'):
            with zipfile.ZipFile(filename, 'r') as z:
                z.extractall('data/')
            os.remove(filename)
            print(f'✅ {filename} açıldı → data/')
        elif filename.endswith('.csv'):
            shutil.move(filename, f'data/{filename}')
            print(f'✅ {filename} → data/')

    !ls -la data/
else:
    # Lokalde data/ zaten var, kontrol et
    !ls -la data/

### Hücre B — Google Drive'dan (Yedek yöntem)

In [ ]:
# ↓ SADECE Yöntem B kullanıyorsan True yap
USE_DRIVE = False

if USE_DRIVE and IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # Drive'daki dosyanın yolunu kendi yoluna göre düzenle:
    drive_zip = '/content/drive/MyDrive/Hepsiburada Product Reviews and Ratings Dataset  .zip'

    import zipfile
    os.makedirs('data', exist_ok=True)
    with zipfile.ZipFile(drive_zip, 'r') as z:
        z.extractall('data/')
    print('✅ Drive\'dan veri açıldı → data/')
    !ls -la data/

### Hücre C — Kaggle API (Gelişmiş)

In [ ]:
# ↓ SADECE Yöntem C kullanıyorsan True yap
USE_KAGGLE_API = False

if USE_KAGGLE_API and IS_COLAB:
    from google.colab import files
    print('📤 kaggle.json dosyasını yükleyin:')
    files.upload()
    !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

    from src.data_loader import download_dataset
    csv_path = download_dataset()
    print(f'✅ İndirildi: {csv_path}')

## 📖 2) Veriyi Pandas ile Oku

In [ ]:
from src.data_loader import load_data, clean_data, add_sentiment_labels
import pandas as pd

csv_files = [f for f in os.listdir('data') if f.endswith('.csv')]
print(f'📁 data/ klasöründeki CSV dosyaları: {csv_files}')

# v2 versiyonu varsa onu tercih et (daha temiz olabilir)
if any('v2' in f for f in csv_files):
    csv_path = 'data/' + [f for f in csv_files if 'v2' in f][0]
else:
    csv_path = 'data/' + csv_files[0]

print(f'📖 Okunuyor: {csv_path}')
df = load_data(csv_path)
df.head()

## 🔍 3) Keşifsel Veri Analizi (EDA)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

print(f'📊 Boyut: {df.shape}')
print(f'📊 Sütunlar: {df.columns.tolist()}')
print(f'\n📊 Veri tipleri:')
print(df.dtypes)
print(f'\n📊 Eksik değerler:')
print(df.isnull().sum())

In [ ]:
df = clean_data(df)
df = add_sentiment_labels(df)
df.head()

In [ ]:
os.makedirs('rapor', exist_ok=True)

# Sınıf dağılımı (Şekil 4.1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sentiment_counts = df['sentiment'].value_counts()
colors_map = {'pozitif': '#28a745', 'nötr': '#ffc107', 'negatif': '#dc3545'}
color_list = [colors_map.get(s, '#888') for s in sentiment_counts.index]

axes[0].pie(sentiment_counts.values, labels=sentiment_counts.index,
            colors=color_list, autopct='%1.1f%%', startangle=90)
axes[0].set_title('Duygu Sınıfı Dağılımı', fontsize=14, fontweight='bold')

axes[1].bar(sentiment_counts.index, sentiment_counts.values, color=color_list)
axes[1].set_title('Yorum Sayıları', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Yorum sayısı')
for i, v in enumerate(sentiment_counts.values):
    axes[1].text(i, v, f'{v:,}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('rapor/sekil_4_1_sinif_dagilimi.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Yorum uzunluk dağılımı (Şekil 4.2)
text_col_candidates = ['review', 'yorum', 'comment', 'text', 'review_text']
text_col = next((c for c in df.columns if c.lower() in text_col_candidates), None)
if text_col is None:
    text_col = df.select_dtypes('object').columns[0]

df['word_count'] = df[text_col].astype(str).str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['word_count'], bins=50, color='#4a90e2', edgecolor='black')
axes[0].set_title('Yorum Uzunluğu Dağılımı', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Kelime sayısı')
axes[0].set_ylabel('Yorum sayısı')
axes[0].axvline(df['word_count'].mean(), color='red', linestyle='--',
                label=f"Ortalama: {df['word_count'].mean():.1f}")
axes[0].legend()

df_plot = df[df['word_count'] < df['word_count'].quantile(0.95)]
sns.boxplot(data=df_plot, x='sentiment', y='word_count',
            palette=colors_map, ax=axes[1])
axes[1].set_title('Sınıf Başına Uzunluk', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Kelime sayısı')

plt.tight_layout()
plt.savefig('rapor/sekil_4_2_uzunluk_dagilimi.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n📊 Uzunluk istatistikleri:')
print(df['word_count'].describe())

## 🧹 4) NLP Ön İşleme

In [ ]:
from src.preprocessing import preprocess_dataframe, get_word_freq

df = preprocess_dataframe(df)
df[[text_col, 'clean_text', 'sentiment']].head(10)

In [ ]:
# En sık kelimeler (Şekil 4.3)
freq = get_word_freq(df, column='clean_text', top_n=20)

plt.figure(figsize=(10, 6))
freq.plot(kind='barh', color='#667eea')
plt.gca().invert_yaxis()
plt.title('En Sık 20 Kelime (Ön İşleme Sonrası)', fontsize=14, fontweight='bold')
plt.xlabel('Frekans')
plt.tight_layout()
plt.savefig('rapor/sekil_4_3_en_sik_kelimeler.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Sınıf bazlı kelimeler
from collections import Counter

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for i, sentiment in enumerate(['pozitif', 'nötr', 'negatif']):
    subset = df[df['sentiment'] == sentiment]['clean_text']
    words = ' '.join(subset.astype(str)).split()
    common = Counter(words).most_common(15)
    if common:
        w, c = zip(*common)
        axes[i].barh(w, c, color=colors_map.get(sentiment, '#888'))
        axes[i].invert_yaxis()
        axes[i].set_title(f'{sentiment.upper()} sınıfında en sık kelimeler',
                          fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('rapor/sekil_4_3b_sinif_bazli_kelimeler.png', dpi=150, bbox_inches='tight')
plt.show()

## 💾 5) Temizlenmiş Veriyi Kaydet

In [ ]:
df = df[df['clean_text'].str.strip().str.len() > 0].reset_index(drop=True)

output_path = 'data/hepsiburada_clean.csv'
df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f'✅ Temizlenmiş veri kaydedildi: {output_path}')
print(f'📊 Toplam {len(df):,} yorum')

if IS_COLAB:
    print('\n💡 Drive\'a yedeklemek için (önerilen):')
    print('   from google.colab import drive')
    print('   drive.mount("/content/drive")')
    print('   !mkdir -p /content/drive/MyDrive/nlp_projesi/data')
    print('   !cp data/hepsiburada_clean.csv /content/drive/MyDrive/nlp_projesi/data/')

## 📋 6) Rapor için Özet İstatistikler

Aşağıdaki çıktıyı **rapor/RAPOR.md** dosyasının **Bölüm 4.3** kısmına ekle.

In [ ]:
print('=' * 60)
print('  📊 RAPOR İÇİN ÖZET İSTATİSTİKLER (Bölüm 4.3)')
print('=' * 60)
print(f'  Toplam yorum sayısı       : {len(df):,}')
print(f'  Ortalama yorum uzunluğu   : {df["word_count"].mean():.1f} kelime')
print(f'  Medyan yorum uzunluğu     : {df["word_count"].median():.0f} kelime')
print(f'  Maksimum yorum uzunluğu   : {df["word_count"].max()} kelime')
print()
print('  Sınıf Dağılımı:')
for sent, count in df['sentiment'].value_counts().items():
    pct = count / len(df) * 100
    print(f'    {sent:10s}: {count:>7,} ({pct:5.1f}%)')

print('\n✅ Notebook 1 tamamlandı! Sonraki adım: 02_baseline_model.ipynb')